In [18]:
import cv2
import numpy as np
import tensorflow as tf
import glob

In [9]:
token_mapping = {
    0: '0', 1: '1', 2: '2', 3: '3', 4: '4', 
    5: '5', 6: '6', 7: '7', 8: '8', 9: '9', 
    10: '+', 11: '-', 12: '*', 13: '/', 14: '=', 
    15: '(', 16: ')', 17: '\\lim', 18: '\\cot'
}

In [10]:
def preprocess_image_for_segmentation(image_path):
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    image = cv2.resize(image, (800, 800))
    _, binary_image = cv2.threshold(image, 180, 255, cv2.THRESH_BINARY_INV)
    return image, binary_image

In [11]:
def segment_image(binary_image, min_contour_size=10):
    contours, _ = cv2.findContours(binary_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    bounding_boxes = [cv2.boundingRect(c) for c in contours if cv2.contourArea(c) > min_contour_size]
    return bounding_boxes

In [12]:
def extract_segments(image, bounding_boxes, target_size=(28,28)):
    segments = []
    for (x, y, w, h) in bounding_boxes:
        segment = image[y:y+h, x:x+w]
        segment = cv2.resize(segment, target_size)
        segments.append(segment)
    return segments

In [ ]:
def synchronous_pipeline(image_path, symbol_model, target_size=(28,28)):
    image, binary_image = preprocess_image_for_segmentation(image_path)
    bounding_boxes = segment_image(binary_image)
    bounding_boxes_sorted = sorted(bounding_boxes, key=lambda b: (b[1], b[0]))
    segments = extract_segments(image, bounding_boxes_sorted, target_size)
    
    predicted_tokens = []
    for seg in segments:
        if len(seg.shape) == 2:
            seg = cv2.cvtColor(seg, cv2.COLOR_GRAY2RGB)
        seg = seg.astype(np.float32) / 255.0
        seg_input = np.expand_dims(seg, axis=0)
        pred = symbol_model.predict(seg_input)
        pred_class = np.argmax(pred, axis=-1)[0]
        token = token_mapping.get(pred_class, '')
        predicted_tokens.append(token)
    predicted_sequence = "".join(predicted_tokens)
    return predicted_sequence

In [8]:
# Example usage:
# image_path = "path/to/your/handwritten_expression.jpg"
# symbol_model = tf.keras.models.load_model("path/to/your/symbol_model.h5")
# latex_sequence = synchronous_pipeline(image_path, symbol_model)
# print("Predicted LaTeX sequence:", latex_sequence)

In [20]:
# def group_bounding_boxes(bounding_boxes, y_threshold=10):
#     # Sort first by y-coordinate
#     bounding_boxes = sorted(bounding_boxes, key=lambda b: b[1])
#     groups = []
#     current_group = []
#     current_y = None
    
#     for box in bounding_boxes:
#         x, y, w, h = box
#         if current_y is None:
#             current_y = y
#             current_group.append(box)
#         else:
#             # If the y difference is within threshold, consider it the same line
#             if abs(y - current_y) < y_threshold:
#                 current_group.append(box)
#             else:
#                 groups.append(sorted(current_group, key=lambda b: b[0]))
#                 current_group = [box]
#                 current_y = y
#     if current_group:
#         groups.append(sorted(current_group, key=lambda b: b[0]))
    
#     # Flatten the list of groups back to a single list in reading order.
#     ordered_boxes = [box for group in groups for box in group]
#     return ordered_boxes

# # Use this function instead of the simple sorted() if needed:
# bounding_boxes_sorted = group_bounding_boxes(bounding_boxes)